<a href="https://colab.research.google.com/github/mdkamrulhasan/data_mining_kdd/blob/main/notebooks/Exploratory_Data_Analysis_COVID19_CIS635.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA): COVID-19 Data

## CIS 635 — Knowledge Discovery and Data Mining

Exploratory Data Analysis (EDA) is the process of **understanding a dataset before building models**. In this notebook, we will apply a repeatable EDA workflow to publicly available COVID-19 data:

1. **Understand the dataset** — rows, columns, feature names, and data types.
2. **Assess data quality** — missing values, duplicate records, and suspicious values.
3. **Summarize variables** — descriptive statistics and frequency counts.
4. **Explore distributions** — histograms, box plots, and other univariate views.
5. **Explore relationships** — grouped summaries, time trends, scatter plots, and correlation.
6. **Perform deeper KDD-oriented analysis** — temporal patterns, rates, subgroup comparisons, and data-quality diagnostics.
7. **Form hypotheses** — use visual and numerical evidence to ask better questions.
8. **Communicate findings** — distinguish association from causation.

### A useful EDA mindset

> **Ask a question → inspect the data → visualize/summarize → interpret → ask a better question.**

EDA is **not** simply producing many plots. Each table or visualization should help answer a specific question about the data.

### Learning goals

By the end of this notebook, you should be able to:

- identify the structure and data types of a real-world health dataset;
- detect and reason about missing and duplicate data;
- distinguish numeric, categorical, and temporal variables;
- use descriptive statistics and frequency tables appropriately;
- identify potential outliers and explain why they require investigation;
- analyze COVID-19 trends over time;
- compare regions and population-normalized measures;
- examine relationships among variables using grouped summaries, plots, and correlation;
- recognize common issues in health/epidemiological data;
- formulate KDD questions and hypotheses from exploratory evidence.

**Dataset:** The notebook is designed to use the three CSV files in the course repository's `data/covid-19` folder. The loading code discovers the files directly from GitHub so the notebook does not depend on hard-coded filenames.


## 1. EDA Workflow

| Stage | Key questions |
|---|---|
| 1. Structure | How many observations and features are present? |
| 2. Data types | Which variables are numeric, categorical, or temporal? |
| 3. Data quality | Are there missing values, duplicates, impossible values, or suspicious observations? |
| 4. Univariate analysis | What does each important variable look like by itself? |
| 5. Bivariate analysis | How does one variable vary with another? |
| 6. Multivariate analysis | Do several variables together reveal a pattern? |
| 7. Temporal analysis | How do cases, deaths, and other measures change over time? |
| 8. Interpretation | What findings are supported by the evidence, and what remains uncertain? |

**Important:** COVID-19 data can contain reporting delays, revisions, missing observations, changes in testing/reporting practices, and differences between geographic reporting systems. EDA should reveal these issues rather than hide them.


## 2. Import Python Libraries

We will primarily use:

- **Pandas** for data manipulation and EDA
- **NumPy** for numerical operations
- **Matplotlib** and **Seaborn** for visualization
- **Datetime utilities** for temporal analysis


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")


## 3. Load the COVID-19 Data from GitHub

The course repository contains the COVID-19 datasets here:

`https://github.com/mdkamrulhasan/data_mining_kdd/tree/main/data/covid-19`

Rather than hard-coding filenames, we clone the repository and discover the CSV files in the folder. This makes the notebook easier to maintain if the files are renamed or updated.

> **Instructor note:** After running this cell, inspect the file list. The intended folder contains the three COVID-19 data files used for this exercise.


In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/mdkamrulhasan/data_mining_kdd.git"
REPO_DIR = "/content/data_mining_kdd"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

DATA_DIR = os.path.join(REPO_DIR, "data", "covid-19")
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

print("COVID-19 data directory:", DATA_DIR)
print("\nCSV files found:")
for f in csv_files:
    print(" -", os.path.basename(f))

print("\nNumber of CSV files:", len(csv_files))


### Inspect the available files

Before loading the data, check filenames and file sizes. This is a useful first step when working with an unfamiliar data repository.


In [ ]:
file_info = pd.DataFrame({
    "file": [os.path.basename(f) for f in csv_files],
    "size_MB": [round(os.path.getsize(f) / (1024**2), 3) for f in csv_files]
})

file_info


## 4. Load the Three COVID-19 Files

The next cell loads all CSV files found in the course `covid-19` directory into a dictionary. We can then inspect each dataset separately before deciding whether they should be merged.

**Why inspect before merging?**

A common KDD mistake is to merge datasets immediately. First determine:

- What does each file represent?
- What are its keys?
- What is its unit of observation?
- What are its time and geographic dimensions?
- Are the same entities represented consistently?


In [ ]:
data = {}

for f in csv_files:
    name = os.path.splitext(os.path.basename(f))[0]
    data[name] = pd.read_csv(f)
    print(f"{name}: {data[name].shape}")


## 5. Initial Inspection

For each dataset, examine the first few rows and its dimensions. At this point, **do not clean anything yet**. The objective is to understand what is actually present.


In [ ]:
for name, df in data.items():
    print("=" * 80)
    print(name)
    print("Shape:", df.shape)
    display(df.head())


## 6. Dataset Structure

### Questions

- How many rows and columns does each dataset contain?
- What does one row represent?
- Which columns identify time?
- Which columns identify a country, state, county, or other region?
- Which variables appear to be measurements or outcomes?

These questions establish the **unit of analysis** before any statistical analysis.


In [ ]:
for name, df in data.items():
    print(f"\n{name}")
    print("-" * len(name))
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Column names:")
    print(list(df.columns))


## 7. Checking Data Types

Pandas may infer some columns as `object`, numeric, or datetime-like values. We need to verify whether those types make sense.

**Pay particular attention to date columns.** A date stored as text can prevent correct temporal analysis.


In [ ]:
for name, df in data.items():
    print(f"\n{name}")
    display(df.dtypes.to_frame("data_type"))


### Convert date-like columns when appropriate

The following helper attempts to identify common date/time column names. It does not force arbitrary columns into datetime format.


In [ ]:
def identify_date_columns(df):
    candidates = []
    for col in df.columns:
        col_lower = col.lower()
        if any(token in col_lower for token in ["date", "time", "day"]):
            candidates.append(col)
    return candidates

for name, df in data.items():
    print(name, "->", identify_date_columns(df))


## 8. Exploring Unique Values

Categorical variables can reveal geographic coverage, reporting levels, and potential inconsistencies.

For each dataset, inspect the number of unique values for columns that appear categorical or identifier-like.


In [ ]:
for name, df in data.items():
    print(f"\n{name}")
    print("=" * 80)
    for col in df.columns:
        if df[col].dtype == "object" or "code" in col.lower() or "name" in col.lower():
            print(f"{col}: {df[col].nunique(dropna=True)} unique values")


### Frequency tables

Frequency counts are especially useful for categorical variables.

Try to identify:

- the most common geographic entities;
- dominant reporting categories;
- possible spelling inconsistencies;
- categories with very few observations.


In [ ]:
for name, df in data.items():
    print(f"\n{name}")
    categorical_cols = df.select_dtypes(include="object").columns.tolist()
    for col in categorical_cols[:5]:
        print(f"\nTop values for {col}:")
        display(df[col].value_counts(dropna=False).head(10))


## 9. Descriptive Statistics

For numeric variables, calculate measures of center, spread, and range.

Look beyond the mean:

- **median** can be more robust when distributions are skewed;
- **standard deviation** describes spread;
- **minimum/maximum** can expose suspicious values;
- **quartiles** help identify unusually large or small observations.


In [ ]:
for name, df in data.items():
    print(f"\n{name}")
    display(df.describe(include="all").T)


## 10. Missing Values

Missing data is common in real-world health datasets.

### Questions

- Which variables contain missing values?
- How much data is missing?
- Is missingness concentrated in particular variables or periods?
- Could the missingness itself contain information about reporting practices?

**Do not automatically replace missing values.** First understand why they are missing.


In [ ]:
for name, df in data.items():
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100
    }).sort_values("missing_percent", ascending=False)

    print(f"\n{name}")
    display(missing[missing["missing_count"] > 0])


## 11. Duplicate Records

Duplicate observations can distort counts and statistics.

Check for:

1. completely duplicated rows;
2. duplicate observations based on likely keys such as `(location, date)`;
3. repeated records caused by multiple geographic/reporting levels.

The correct definition of a duplicate depends on the dataset's **unit of observation**.


In [ ]:
for name, df in data.items():
    print(f"{name}:")
    print("  Complete duplicate rows:", df.duplicated().sum())


## 12. Data-Quality Checks

COVID-19 datasets often contain cumulative counts and derived measures. We should look for values that deserve investigation, such as:

- negative counts;
- impossible percentages;
- cumulative values that decrease;
- inconsistent date ordering;
- unexpectedly large jumps.

A suspicious value is **not automatically an error**. Revisions and reporting changes can produce unusual observations.


In [ ]:
for name, df in data.items():
    numeric = df.select_dtypes(include=np.number)
    negative_counts = (numeric < 0).sum().sort_values(ascending=False)

    print(f"\n{name} — negative numeric values")
    display(negative_counts[negative_counts > 0])


## 13. Temporal Variables

For datasets containing dates, convert the date field and inspect:

- earliest date;
- latest date;
- number of unique dates;
- gaps in the time series.

Temporal structure is critical because COVID-19 data is fundamentally time-dependent.


In [ ]:
date_columns = {}

for name, df in data.items():
    cols = identify_date_columns(df)
    if cols:
        date_col = cols[0]
        date_columns[name] = date_col
        data[name][date_col] = pd.to_datetime(data[name][date_col], errors="coerce")
        print(f"{name}: using '{date_col}' as the primary date column")
        print("  Earliest:", data[name][date_col].min())
        print("  Latest:  ", data[name][date_col].max())
        print("  Unique dates:", data[name][date_col].nunique())


## 14. Distribution of Numeric Variables

Histograms help us understand the shape of numeric variables.

Look for:

- skewness;
- heavy tails;
- multiple modes;
- concentration near zero;
- extreme observations.

For COVID-19 counts, strong right skew is often expected because regions differ greatly in population and outbreak size.


In [ ]:
for name, df in data.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    for col in numeric_cols[:6]:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[col].dropna(), kde=True)
        plt.title(f"{name}: Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.show()


## 15. Box Plots and Potential Outliers

Box plots provide a compact view of the median, quartiles, and potential extreme values.

Remember:

> **An outlier is an observation that deserves investigation—not necessarily an observation that should be deleted.**

In epidemiological data, a sudden increase may reflect a real outbreak, a reporting backlog, or a change in reporting methodology.


In [ ]:
for name, df in data.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    for col in numeric_cols[:5]:
        plt.figure(figsize=(8, 3.5))
        sns.boxplot(x=df[col].dropna())
        plt.title(f"{name}: Box Plot of {col}")
        plt.xlabel(col)
        plt.show()


## 16. COVID-19 Trends Over Time

Now we move from univariate analysis to a central health-informatics question:

### **How did the reported COVID-19 burden change over time?**

For a dataset with a date column, identify cumulative case/death-like variables and visualize them for selected regions where possible.

The exact column names are discovered from the data rather than assumed.


In [ ]:
for name, df in data.items():
    if name not in date_columns:
        continue

    date_col = date_columns[name]
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    print(f"\n{name}")
    print("Date column:", date_col)
    print("Candidate numeric variables:")
    print(numeric_cols)


### A reusable time-series plot

The next cell selects a likely COVID outcome column based on column names and plots its aggregate value by date. Because the datasets may have different structures, inspect the output before interpreting it.


In [ ]:
def choose_outcome_column(df):
    preferred = [
        "confirmed", "cases", "cumulative_confirmed",
        "deaths", "cumulative_deceased", "cumulative_deaths",
        "recovered", "vaccines", "people_vaccinated"
    ]
    lower_map = {c.lower(): c for c in df.columns}
    for p in preferred:
        if p in lower_map:
            return lower_map[p]

    keywords = ["case", "death", "recover", "vaccin", "hospital"]
    for c in df.select_dtypes(include=np.number).columns:
        if any(k in c.lower() for k in keywords):
            return c
    return None

for name, df in data.items():
    if name not in date_columns:
        continue

    outcome = choose_outcome_column(df)
    if outcome is None:
        continue

    ts = df.groupby(date_columns[name], dropna=True)[outcome].sum().reset_index()

    plt.figure(figsize=(11, 4.5))
    sns.lineplot(data=ts, x=date_columns[name], y=outcome)
    plt.title(f"{name}: Aggregate {outcome} Over Time")
    plt.xlabel("Date")
    plt.ylabel(outcome)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 17. Comparing Regions

Aggregate totals can be misleading when regions have very different populations.

A useful EDA question is:

### **Which regions experienced the largest reported burden?**

Start with raw counts, then consider population-normalized measures if population data is available.


In [ ]:
for name, df in data.items():
    if name not in date_columns:
        continue

    outcome = choose_outcome_column(df)
    if outcome is None:
        continue

    # Look for a likely geographic name column.
    geo_candidates = [
        c for c in df.columns
        if any(k in c.lower() for k in ["country", "state", "province", "region", "location", "county"])
    ]

    print(f"\n{name}")
    print("Outcome:", outcome)
    print("Possible geographic columns:", geo_candidates)


## 18. Population-Normalized Analysis

When population is available, compare rates rather than only counts.

For example:

\[
\text{Cases per 100,000} =
\frac{\text{Cases}}{\text{Population}} \times 100,000
\]

This illustrates an important KDD principle:

> **The choice of representation can change the pattern you discover.**

A large population does not necessarily imply a high per-capita burden.


In [ ]:
for name, df in data.items():
    pop_cols = [c for c in df.columns if "population" in c.lower()]
    outcome = choose_outcome_column(df)

    if pop_cols and outcome:
        pop = pop_cols[0]
        temp = df[[outcome, pop]].copy()
        temp["rate_per_100k"] = temp[outcome] / temp[pop] * 100000

        print(f"{name}: {outcome} normalized by {pop}")
        display(temp["rate_per_100k"].describe())


## 19. Grouped Analysis

Grouped summaries allow us to ask questions such as:

- Do outcomes differ across geographic groups?
- How variable are outcomes within a region?
- Which regions have unusually high or low values?
- Does the pattern change over time?

Use `groupby()` to move from raw observations to interpretable summaries.


In [ ]:
for name, df in data.items():
    outcome = choose_outcome_column(df)
    if outcome is None:
        continue

    geo_candidates = [
        c for c in df.columns
        if any(k in c.lower() for k in ["country", "state", "province", "region", "location", "county"])
    ]

    if geo_candidates:
        geo = geo_candidates[0]
        summary = (
            df.groupby(geo)[outcome]
              .agg(["count", "mean", "median", "max"])
              .sort_values("mean", ascending=False)
              .head(15)
        )
        print(f"\n{name}: {outcome} by {geo}")
        display(summary)


## 20. Correlation Analysis

Correlation can help identify variables that move together.

However:

- correlation measures association, not causation;
- cumulative variables can be strongly correlated simply because they both increase over time;
- missing values and extreme observations can affect correlations;
- correlation does not establish a causal relationship.

Use correlation as a **question-generating tool**, not as proof of causality.


In [ ]:
for name, df in data.items():
    numeric = df.select_dtypes(include=np.number)

    if numeric.shape[1] >= 2:
        corr = numeric.corr(numeric_only=True)

        plt.figure(figsize=(10, 7))
        sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
        plt.title(f"{name}: Correlation Matrix")
        plt.tight_layout()
        plt.show()


## 21. Pairwise Relationships

Choose two meaningful numeric variables and examine their relationship using a scatter plot.

For COVID-19 data, useful questions may involve:

- cases vs. deaths;
- cases vs. hospitalizations;
- cases vs. tests;
- vaccination vs. reported outcomes;

**Only use variables that actually exist in the selected dataset.**


In [ ]:
for name, df in data.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    if len(numeric_cols) >= 2:
        x, y = numeric_cols[:2]

        plt.figure(figsize=(7, 5))
        sns.scatterplot(data=df, x=x, y=y, alpha=0.5)
        plt.title(f"{name}: {x} vs. {y}")
        plt.tight_layout()
        plt.show()


## 22. Deeper KDD Analysis: Daily Changes

Cumulative counts are often easier to interpret after calculating daily changes.

For a cumulative measure \(C_t\):

\[
\Delta C_t = C_t - C_{t-1}
\]

This transformation can reveal outbreak peaks that are difficult to see in a cumulative curve.


In [ ]:
for name, df in data.items():
    if name not in date_columns:
        continue

    outcome = choose_outcome_column(df)
    if outcome is None:
        continue

    temp = df.sort_values(date_columns[name]).copy()
    temp["daily_change"] = temp[outcome].diff()

    plt.figure(figsize=(11, 4.5))
    sns.lineplot(data=temp, x=date_columns[name], y="daily_change")
    plt.title(f"{name}: Change in {outcome} Over Time")
    plt.xlabel("Date")
    plt.ylabel(f"Daily change in {outcome}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 23. Rolling Averages

Daily reporting can be noisy. A rolling average can help reveal the underlying trend.

For example, a 7-day rolling average is often useful for illustrating how smoothing changes interpretation.

**EDA question:** What patterns become easier to see after smoothing? What information might smoothing hide?


In [ ]:
for name, df in data.items():
    if name not in date_columns:
        continue

    outcome = choose_outcome_column(df)
    if outcome is None:
        continue

    temp = df.sort_values(date_columns[name]).copy()
    temp["daily_change"] = temp[outcome].diff()
    temp["rolling_7"] = temp["daily_change"].rolling(7).mean()

    plt.figure(figsize=(11, 4.5))
    sns.lineplot(data=temp, x=date_columns[name], y="rolling_7")
    plt.title(f"{name}: 7-Day Rolling Average of Daily Change in {outcome}")
    plt.xlabel("Date")
    plt.ylabel("7-day rolling average")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 24. Subgroup Analysis

Health data often becomes more informative when examined by subgroup.

Depending on the available variables, consider:

- country or state;
- age group;
- sex;
- geographic reporting level;
- vaccination status;
- hospitalization status.

**Important:** Do not assume a subgroup variable exists. Inspect the actual schema first.


In [ ]:
for name, df in data.items():
    possible_subgroups = [
        c for c in df.columns
        if any(k in c.lower() for k in ["age", "sex", "gender", "state", "country", "region", "status"])
    ]

    print(f"{name}:")
    print(possible_subgroups)


## 25. Interpreting Association Carefully

Suppose we observe that two variables are highly correlated.

Before claiming a relationship, ask:

1. Could both variables simply be increasing over time?
2. Could population size explain the association?
3. Could testing/reporting practices affect the observed relationship?
4. Are there important confounding variables?
5. Is the relationship consistent across regions and time periods?

This is especially important in observational health data.


## 26. KDD Perspective: From EDA to Modeling

EDA prepares us for later KDD tasks.

Potential downstream tasks include:

- **classification:** predict whether a region will experience a high-burden period;
- **regression:** predict future case/death counts;
- **clustering:** group regions according to epidemic trajectories;
- **anomaly detection:** identify unusual reporting patterns;
- **time-series forecasting:** predict future outcomes;
- **feature engineering:** construct rates, rolling averages, lag variables, and growth measures.

The goal of EDA is to understand the data well enough to make these modeling decisions responsibly.


# Student Exercises

The exercises below are intentionally more open-ended than the worked examples.

### Exercise 1 — Dataset Understanding
Choose one of the three COVID-19 datasets.

1. What is the unit of observation?
2. How many rows and columns are present?
3. Identify numeric, categorical, and temporal variables.
4. Which variables appear most important for a future predictive model?
5. Explain why.

### Exercise 2 — Data Quality
For your selected dataset:

1. Identify the five columns with the largest number of missing values.
2. Determine whether any duplicate records exist.
3. Search for negative or otherwise suspicious numeric values.
4. Select one data-quality issue and explain how you would investigate it.

### Exercise 3 — Temporal Exploration
Choose one COVID-19 outcome.

1. Plot its cumulative trend over time.
2. Calculate its daily change.
3. Plot the daily change.
4. Add a 7-day rolling average.
5. Identify at least two interesting temporal patterns.

### Exercise 4 — Geographic Comparison
Select a geographic variable.

1. Identify the 10 regions with the largest reported outcome.
2. Compare raw counts with population-normalized rates if population is available.
3. Explain why the ranking may change after normalization.

### Exercise 5 — Correlation Is Not Causation
Find two variables with a relatively strong correlation.

1. Calculate their correlation.
2. Create an appropriate visualization.
3. Provide at least two possible explanations for the association.
4. Explain why the evidence does **not** establish causation.

### Exercise 6 — Outlier Investigation
Identify one potential outlier.

1. Locate the observation.
2. Determine when and where it occurred.
3. Examine related variables.
4. Decide whether it appears to be a data error, a reporting artifact, or a potentially meaningful observation.
5. Justify your conclusion.

### Exercise 7 — Mini KDD Investigation
Formulate **one research question** that could be investigated using the COVID-19 data.

Follow the workflow:

> **Question → Data inspection → Cleaning/Transformation → Visualization → Finding → Interpretation → New question**

Your answer should include at least **two visualizations** and one numerical summary.


# Final Reflection

Before leaving the notebook, answer these questions:

1. What was the most important data-quality issue you discovered?
2. What was the most interesting pattern in the data?
3. Which visualization communicated your finding most effectively?
4. What additional data would make the analysis more informative?
5. What predictive or descriptive KDD problem would you investigate next?

### Key takeaway

> **Good EDA does not end with a chart. It ends with a better understanding of the data and better questions for the next stage of knowledge discovery.**


## References and Data Source

The notebook uses the COVID-19 datasets maintained in the course repository:

- Course repository: `https://github.com/mdkamrulhasan/data_mining_kdd`
- COVID-19 data folder: `https://github.com/mdkamrulhasan/data_mining_kdd/tree/main/data/covid-19`

The notebook also follows the EDA workflow of the provided automobile EDA notebook, including dataset inspection, data types, unique values, descriptive statistics, missing-value analysis, outlier exploration, visualization, relationships, interpretation, and student exercises.
